# 09 — Descriptivo clínico de cohorte y desenlaces

Descripción agregada de la cohorte, los desenlaces retrospectivos y su temporalidad sobre `development` y `validation`. El test no se carga ni se resume. Las celdas pequeñas se suprimen y no se realizan pruebas de significación; las diferencias estandarizadas describen balance, no seleccionan variables. Las cifras del demo son controles técnicos, no estimaciones clínicas.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, tempfile
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.descriptive import (categorical_descriptive_summary, continuous_descriptive_summary, outcome_burden_summary, outcome_timing_summary, partition_standardized_differences, prepare_descriptive_cohort)
from scripts.build_demo_sofa_incremental import canonical_config
config = canonical_config()
candidates = sorted((PROJECT_ROOT / 'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json'))
valid = []
for manifest_path in candidates:
    store = ArtifactStore(manifest_path.parent)
    try:
        manifest = store.validate('sepsis3_development_features', expected_config=config)
        valid.append((manifest.created_at_utc, manifest_path.parents[1]))
    except (FileNotFoundError, ArtifactValidationError):
        pass
if not valid:
    raise RuntimeError('No hay artefactos compatibles. Ejecute: python scripts/build_demo_sofa_incremental.py --stage all --resume')
RUN_ROOT = sorted(valid, key=lambda item: (item[0], str(item[1])))[-1][1]
cohort_store = ArtifactStore(RUN_ROOT / '00_cohort')
label_store = ArtifactStore(RUN_ROOT / '40_labels')
cohort = cohort_store.read_dataframe('cohort_stays', expected_config=config)
sepsis_stays = label_store.read_dataframe('sepsis_stays', expected_config=config)
shock_stays = label_store.read_dataframe('septic_shock_stays', expected_config=config)
clinical = prepare_descriptive_cohort(cohort, sepsis_stays, shock_stays, split_config=config['splits'])
assert set(clinical['partition']).issubset({'development', 'validation'})
print(f'Ejecución validada: {RUN_ROOT.name}; estancias descriptivas: {len(clinical)}')

## Tabla 1, flujo y carga de desenlaces

Los recuentos se expresan por paciente, ingreso y estancia. Las variables continuas se resumen mediante mediana e IQR. Las categorías administrativas se describen sin interpretación biológica. `privacy_suppressed=True` significa que la celda y, cuando corresponde, una celda complementaria se ocultan.

In [ ]:
privacy_minimum = int(json.loads((PROJECT_ROOT / 'config/subgroups.json').read_text())['privacy_minimum_cell'])
burden = outcome_burden_summary(clinical, privacy_minimum_cell=privacy_minimum)
continuous = continuous_descriptive_summary(clinical, privacy_minimum_cell=privacy_minimum)
categorical = categorical_descriptive_summary(clinical, privacy_minimum_cell=privacy_minimum)
standardized = partition_standardized_differences(clinical, privacy_minimum_cell=privacy_minimum)
timing = outcome_timing_summary(clinical, sepsis_stays, shock_stays, privacy_minimum_cell=privacy_minimum)
display(burden, continuous, categorical, standardized, timing)
table_dir = PROJECT_ROOT / 'data/derived/reports/tables'
table_dir.mkdir(parents=True, exist_ok=True)
for name, table in {'09_outcome_burden': burden, '09_continuous_table1': continuous, '09_categorical_table1': categorical, '09_partition_smd': standardized, '09_outcome_timing': timing}.items():
    table.to_csv(table_dir / f'{name}.csv', index=False)

## Figuras descriptivas con ggplot2

Solo se grafican filas no suprimidas. Las barras de desenlaces usan estancias como denominador; los intervalos de las variables continuas son IQR, no intervalos de confianza.

In [ ]:
rscript = shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible en el entorno.')
figure_dir = PROJECT_ROOT / 'data/derived/reports/figures'
figure_dir.mkdir(parents=True, exist_ok=True)
burden_plot = burden.query('privacy_suppressed == False').copy()
continuous_plot = continuous.query("stratum == 'partition' and privacy_suppressed == False").copy()
smd_plot = standardized.query('privacy_suppressed == False').copy()
smd_plot['absolute_standardized_difference'] = smd_plot['standardized_difference'].abs()
r_code = r'''
args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
b <- read.csv(args[1]); c <- read.csv(args[2]); s <- read.csv(args[3])
p1 <- ggplot(b, aes(sample, stay_percent, fill=endpoint)) + geom_col(position='dodge') + scale_y_continuous(limits=c(0,100)) + labs(title='Carga de desenlaces por estancia', x=NULL, y='Estancias con desenlace (%)', fill='Desenlace') + theme_minimal(base_size=11)
p2 <- ggplot(c, aes(level, median, colour=variable, ymin=q1, ymax=q3)) + geom_pointrange(position=position_dodge(width=.4)) + labs(title='Variables continuas por partición', subtitle='Mediana e IQR', x=NULL, y='Valor', colour='Variable') + theme_minimal(base_size=11)
p3 <- ggplot(s, aes(absolute_standardized_difference, reorder(paste(variable, level), absolute_standardized_difference))) + geom_point(colour='#2878B5') + geom_vline(xintercept=.1, linetype=2, colour='#D9534F') + labs(title='Balance development–validation', subtitle='Diferencia estandarizada absoluta; línea en 0,10 descriptiva', x='|Diferencia estandarizada|', y=NULL) + theme_minimal(base_size=10)
ggsave(args[4], p1, width=8, height=4.8, device=grDevices::svg)
ggsave(args[5], p2, width=8, height=4.8, device=grDevices::svg)
ggsave(args[6], p3, width=9, height=6, device=grDevices::svg)
'''
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp); b_csv = tmp/'burden.csv'; c_csv = tmp/'continuous.csv'; s_csv = tmp/'smd.csv'
    burden_plot.to_csv(b_csv, index=False); continuous_plot.to_csv(c_csv, index=False); smd_plot.to_csv(s_csv, index=False)
    figures = [figure_dir/'09_outcome_burden.svg', figure_dir/'09_continuous_table1.svg', figure_dir/'09_partition_balance.svg']
    run = subprocess.run([rscript, '-e', r_code, str(b_csv), str(c_csv), str(s_csv), *map(str, figures)], capture_output=True, text=True)
    if run.returncode: raise RuntimeError(run.stderr)
display(*(SVG(filename=str(path)) for path in figures))

## Interpretación y criterio para avanzar

Este descriptivo caracteriza la población y documenta balance entre development y validation; no evalúa causalidad ni permite seleccionar variables, puntos de corte o modelos. Sepsis-3 y shock son fenotipos retrospectivos. Las tablas completas solo serán interpretables después de ejecutar MIMIC-IV completo y cerrar las decisiones clínicas pendientes.